[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Generative_Autoencoder_From_Scratch.ipynb)

# Autoencoder From Scratch — Every Gradient by Hand

**Companion notebook to** [`playbooks/ai/generative/02_Concepts.md`](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/generative/02_Concepts.md).

This notebook trains a vanilla autoencoder **using only NumPy** — no PyTorch autograd, no high-level libraries. Every weight, every gradient, every update is computed by hand.

**Why this matters.** Autoencoders are the conceptual foundation for VAE, GAN encoders, and diffusion U-Nets. If you can train an AE by hand, the rest of the generative family becomes mechanical extensions of the same pattern.

## What you will see
1. Build a 4 → 2 → 4 autoencoder from scratch
2. Forward pass: encode + decode on a single example
3. Compute reconstruction loss
4. Compute every gradient by hand: decoder, then through latent, then encoder
5. **Verify** that PyTorch autograd produces the same numbers
6. Train for 100 cycles — watch the loss converge to zero
7. Train on a small dataset of 4 examples
8. Visualize the 2-dim latent space

By the end, `nn.Linear` + `loss.backward()` for any encoder-decoder network will stop being magic.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("NumPy:", np.__version__)

## 1. The Architecture

```
Input x (4-dim)
   ↓ Encoder W₁ (2×4) + b₁ (2)
z₁ = W₁·x + b₁
h = ReLU(z₁)              ← 2-dim latent code (the bottleneck)
   ↓ Decoder W₂ (4×2) + b₂ (4)
x̂ = W₂·h + b₂              ← reconstruction
   ↓
L = 0.5 · ||x̂ − x||²        ← MSE
```

Total parameters: `(W₁: 8) + (b₁: 2) + (W₂: 8) + (b₂: 4) = 22`.

## 2. Initial Values

In [ ]:
# Single training example
x = np.array([1.0, 0.5, 0.0, 0.5], dtype=np.float32)

# Encoder weights
W1 = np.array([[ 0.5,  0.2, -0.3,  0.1],
               [-0.1,  0.3,  0.4, -0.2]], dtype=np.float32)
b1 = np.array([0.0, 0.0], dtype=np.float32)

# Decoder weights
W2 = np.array([[ 0.4,  0.3],
               [-0.2,  0.5],
               [ 0.1, -0.4],
               [ 0.3,  0.2]], dtype=np.float32)
b2 = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)

alpha = 0.1   # Learning rate

print(f"Input x: {x}")
print(f"W1 (encoder, 2x4):\n{W1}")
print(f"b1: {b1}")
print(f"W2 (decoder, 4x2):\n{W2}")
print(f"b2: {b2}")
print(f"alpha: {alpha}")

## 3. Forward Pass — Encode then Decode

The encoder compresses 4 dims to 2 (a non-trivial information loss). The decoder reconstructs back to 4.

In [ ]:
def forward(x, W1, b1, W2, b2):
    z1 = W1 @ x + b1                  # Linear: 4 → 2
    h  = np.maximum(0, z1)            # ReLU activation (the bottleneck code)
    x_recon = W2 @ h + b2             # Linear: 2 → 4 (no activation — regression)
    return z1, h, x_recon

z1, h, x_recon = forward(x, W1, b1, W2, b2)
print(f"z1 (pre-activation): {z1}")
print(f"h (after ReLU): {h}")
print(f"x_recon: {x_recon}")
print(f"x:        {x}     (target — the input itself!)")

**Notice.** `z1[1] = -0.05 < 0`, so `h[1] = 0`. The second neuron in the bottleneck is dead. It will receive zero gradient in the backward pass.

## 4. Reconstruction Loss

In [ ]:
def loss_fn(x_recon, x):
    diff = x_recon - x
    return 0.5 * (diff**2).sum()

L = loss_fn(x_recon, x)
diff = x_recon - x
print(f"diff = x_recon - x = {diff}")
print(f"L = 0.5 · sum(diff²) = {L:.4f}")

## 5. Backward Pass — Through Two Networks

Walk the chain backward from the loss to the encoder weights. Two key features:
1. **Gradients flow through both networks** — encoder learns from feedback that traveled through the entire decoder
2. **Dead ReLU silences whole branches** — `h[1] = 0` means no decoder weights connecting from neuron 1 get updated, AND no encoder weights into neuron 1 get updated

### 5.1 Gradient Seed and Decoder Gradients

In [ ]:
# Seed: ∂L/∂x_recon = x_recon - x
dL_dx_recon = x_recon - x
print(f"∂L/∂x_recon = {dL_dx_recon}")

# Decoder weights: x_recon[i] = sum_j W2[i,j]·h[j] + b2[i]
# So ∂L/∂W2[i,j] = ∂L/∂x_recon[i] · h[j]
dL_dW2 = np.outer(dL_dx_recon, h)
dL_db2 = dL_dx_recon.copy()

print(f"∂L/∂W2 = ∂L/∂x_recon ⊗ h:\n{dL_dW2}")
print(f"∂L/∂b2 = {dL_db2}")

**Notice.** The right column of `∂L/∂W2` is zero because `h[1] = 0`. **Dead ReLU killed all decoder weights connecting from neuron 1.**

### 5.2 Backward Through the Latent

`∂L/∂h = W2ᵀ · ∂L/∂x_recon` — gradients pass back through the decoder weights into the latent.

In [ ]:
dL_dh = W2.T @ dL_dx_recon
print(f"∂L/∂h = W2ᵀ · ∂L/∂x_recon = {dL_dh}")

### 5.3 Through the Encoder ReLU

In [ ]:
# ∂h/∂z1 = 1 if z1 > 0, else 0
relu_mask = (z1 > 0).astype(np.float32)
dL_dz1 = dL_dh * relu_mask

print(f"ReLU mask: {relu_mask}")
print(f"∂L/∂z1 = ∂L/∂h ⊙ ReLU' = {dL_dz1}")

**Notice.** `∂L/∂z1[1] = 0` because the ReLU killed neuron 1. The encoder will not update any weights leading INTO this neuron.

### 5.4 Encoder Gradients

In [ ]:
# Encoder weights: z1[j] = sum_i W1[j,i]·x[i] + b1[j]
# So ∂L/∂W1[j,i] = ∂L/∂z1[j] · x[i]
dL_dW1 = np.outer(dL_dz1, x)
dL_db1 = dL_dz1.copy()

print(f"∂L/∂W1:\n{dL_dW1}")
print(f"∂L/∂b1 = {dL_db1}")

**Notice.** The bottom row of `∂L/∂W1` is all zeros. **Dead ReLU on `z1[1]` blocked gradient from flowing into ALL of `W1`'s second row.** Both incoming and outgoing weights to a dead neuron stop learning until the neuron revives.

This is why dying ReLU is a serious problem: in a small network, a few dead neurons can paralyze whole subspaces of the model.

## 6. Verify Against PyTorch Autograd

Set up the same computation in PyTorch with `requires_grad=True`. Run the forward and `.backward()`. Compare gradients.

In [ ]:
import torch

# Create PyTorch tensors with grad tracking on the parameters
W1_t = torch.tensor(W1, requires_grad=True)
b1_t = torch.tensor(b1, requires_grad=True)
W2_t = torch.tensor(W2, requires_grad=True)
b2_t = torch.tensor(b2, requires_grad=True)
x_t  = torch.tensor(x, requires_grad=False)

# Forward — same equations
z1_t = W1_t @ x_t + b1_t
h_t  = torch.relu(z1_t)
x_recon_t = W2_t @ h_t + b2_t
L_t = 0.5 * ((x_recon_t - x_t)**2).sum()

# Backward
L_t.backward()

print("PyTorch autograd gradients (should match our manual numbers):")
print(f"\n∂L/∂W1:\n{W1_t.grad.numpy()}")
print(f"Manual ∂L/∂W1:\n{dL_dW1}")
print(f"Match: {np.allclose(W1_t.grad.numpy(), dL_dW1)}")
print()
print(f"∂L/∂b1: {b1_t.grad.numpy()}")
print(f"Manual: {dL_db1}")
print(f"Match: {np.allclose(b1_t.grad.numpy(), dL_db1)}")
print()
print(f"∂L/∂W2:\n{W2_t.grad.numpy()}")
print(f"Manual ∂L/∂W2:\n{dL_dW2}")
print(f"Match: {np.allclose(W2_t.grad.numpy(), dL_dW2)}")
print()
print(f"∂L/∂b2: {b2_t.grad.numpy()}")
print(f"Manual: {dL_db2}")
print(f"Match: {np.allclose(b2_t.grad.numpy(), dL_db2)}")

**All four sets of gradients identical.** PyTorch's autograd is doing the same chain rule we just walked through by hand — it just does it on bigger networks.

## 7. Update Step

In [ ]:
W1_new = W1 - alpha * dL_dW1
b1_new = b1 - alpha * dL_db1
W2_new = W2 - alpha * dL_dW2
b2_new = b2 - alpha * dL_db2

print(f"W1_new:\n{W1_new}")
print(f"b1_new: {b1_new}")
print(f"W2_new:\n{W2_new}")
print(f"b2_new: {b2_new}")

## 8. Verify Loss Drops in Cycle 2

In [ ]:
z1_c2, h_c2, x_recon_c2 = forward(x, W1_new, b1_new, W2_new, b2_new)
L_c2 = loss_fn(x_recon_c2, x)

print(f"Cycle 1 loss: {L:.4f}")
print(f"Cycle 2 loss: {L_c2:.4f}")
print(f"Improvement:  {L - L_c2:+.4f}")
print()
print(f"Cycle 1 reconstruction: {x_recon}")
print(f"Cycle 2 reconstruction: {x_recon_c2}")
print(f"Target:                 {x}")

**Loss dropped from 0.521 to 0.366 in one step.** Reconstruction is improving. Let's run many more cycles.

## 9. Train for 100 Cycles — Watch It Converge

Train on the same single example. Loss should converge to near-zero (the model memorizes one input perfectly).

In [ ]:
# Reset to initial weights
W1c, b1c = W1.copy(), b1.copy()
W2c, b2c = W2.copy(), b2.copy()

losses = []
EPOCHS = 100

for step in range(EPOCHS):
    # Forward
    z1, h, x_recon = forward(x, W1c, b1c, W2c, b2c)
    L = loss_fn(x_recon, x)
    losses.append(L)

    # Backward
    dL_dx_recon = x_recon - x
    dL_dW2 = np.outer(dL_dx_recon, h)
    dL_db2 = dL_dx_recon.copy()
    dL_dh = W2c.T @ dL_dx_recon
    dL_dz1 = dL_dh * (z1 > 0)
    dL_dW1 = np.outer(dL_dz1, x)
    dL_db1 = dL_dz1.copy()

    # Update
    W1c -= alpha * dL_dW1
    b1c -= alpha * dL_db1
    W2c -= alpha * dL_dW2
    b2c -= alpha * dL_db2

print(f"Step  0: L = {losses[0]:.6f}")
print(f"Step 10: L = {losses[10]:.6f}")
print(f"Step 50: L = {losses[50]:.6f}")
print(f"Step 99: L = {losses[-1]:.8f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Reconstruction loss')
plt.title('Autoencoder training — single example')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

print(f"Final reconstruction: {x_recon}")
print(f"Target:                {x}")

## 10. Train on a Small Dataset

Memorizing one input is trivial. Let's train on 4 different inputs — the model must learn a *shared* compression scheme that works for all of them.

In [ ]:
# Small dataset: 4 inputs
X_data = np.array([
    [1.0, 0.5, 0.0, 0.5],
    [0.5, 1.0, 0.5, 0.0],
    [0.0, 0.5, 1.0, 0.5],
    [0.5, 0.0, 0.5, 1.0],
], dtype=np.float32)

# Reset to initial weights
W1d, b1d = W1.copy(), b1.copy()
W2d, b2d = W2.copy(), b2.copy()

losses_dataset = []
EPOCHS = 500

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for x_i in X_data:
        # Forward
        z1, h, x_recon = forward(x_i, W1d, b1d, W2d, b2d)
        epoch_loss += loss_fn(x_recon, x_i)

        # Backward
        dL_dx_recon = x_recon - x_i
        dL_dW2 = np.outer(dL_dx_recon, h)
        dL_db2 = dL_dx_recon.copy()
        dL_dh = W2d.T @ dL_dx_recon
        dL_dz1 = dL_dh * (z1 > 0)
        dL_dW1 = np.outer(dL_dz1, x_i)
        dL_db1 = dL_dz1.copy()

        # Update (per-example update — SGD)
        W1d -= alpha * dL_dW1
        b1d -= alpha * dL_db1
        W2d -= alpha * dL_dW2
        b2d -= alpha * dL_db2

    losses_dataset.append(epoch_loss / len(X_data))

print(f"Loss at epoch 1:   {losses_dataset[0]:.4f}")
print(f"Loss at epoch 50:  {losses_dataset[49]:.4f}")
print(f"Loss at epoch 100: {losses_dataset[99]:.4f}")
print(f"Loss at epoch 500: {losses_dataset[-1]:.6f}")
print()
print("Reconstructions after training:")
for x_i in X_data:
    _, _, x_recon = forward(x_i, W1d, b1d, W2d, b2d)
    print(f"  input: {x_i}  ->  recon: {x_recon}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses_dataset)
plt.xlabel('Epoch')
plt.ylabel('Average reconstruction loss')
plt.title('Autoencoder training on 4-example dataset')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

## 11. Visualize the 2-Dim Latent Space

The bottleneck is 2-dim, so we can plot it. Each input maps to a single 2-dim point. Similar inputs should be near each other in latent space — that is what compression *means*.

In [ ]:
# Encode each training example
encoded = np.array([forward(x_i, W1d, b1d, W2d, b2d)[1] for x_i in X_data])

plt.figure(figsize=(7, 6))
for i, (e, x_i) in enumerate(zip(encoded, X_data)):
    plt.scatter(e[0], e[1], s=200)
    plt.annotate(f'  x_{i}: {x_i.tolist()}', (e[0], e[1]), fontsize=10)

plt.xlabel('Latent dimension 1 (h[0])')
plt.ylabel('Latent dimension 2 (h[1])')
plt.title('Encoded inputs in 2-dim latent space')
plt.grid(True, alpha=0.3)
plt.axhline(0, color='gray', alpha=0.3)
plt.axvline(0, color='gray', alpha=0.3)
plt.show()

print("Latent codes:")
for i, e in enumerate(encoded):
    print(f"  x_{i} -> latent {e}")

**This 2D plot is the latent space.** Each input is one point. The decoder learns to map each point back to its original 4-dim input. The encoder learned to *organize* the 4 inputs into a 2-dim representation. Walk linearly between two points and the decoder produces a smooth interpolation between the corresponding outputs.

This same idea — but with a smarter latent space — becomes:
- **VAE** (next section in the markdown): the latent is a *distribution*, not a point. The KL term forces it to be a tractable distribution.
- **GAN**: the latent is sampled directly from `N(0, 1)` and the decoder = generator. No encoder.
- **Diffusion**: the latent IS the noise space, same shape as the output. No bottleneck.

All three families are variations on the encode-decode pattern you just trained by hand.

## What you just did

| Step | What you proved |
|---|---|
| Forward pass | Encoder + decoder is just two `Linear + activation` layers stacked |
| Loss | Reconstruction loss makes the AE unsupervised — the target is the input itself |
| Backward through decoder | Standard chain rule for a linear layer |
| Backward through latent | `W2ᵀ · dL/dx_recon` passes gradient through the decoder weights |
| Backward through encoder | Standard chain rule for a linear layer + ReLU |
| **Two-network gradient flow** | The encoder learns from gradients that traveled through the entire decoder. This is what makes end-to-end training work. |
| **Dead ReLU silences subspaces** | A dead bottleneck neuron blocks both incoming and outgoing gradients. With small bottlenecks, this is catastrophic. |
| PyTorch autograd verification | Manual gradients exactly match `loss.backward()` on equivalent code |
| 100-step training | Single example loss drops to ~0 (memorization) |
| 4-example training | Multi-example loss converges to a non-zero floor (compression budget) |
| 2D latent visualization | The bottleneck is the geometry of compressed information |

When you write `nn.Linear` + `loss.backward()` for any encoder-decoder network in PyTorch — autoencoder, VAE encoder, U-Net, GAN generator/discriminator — you now know exactly what is happening underneath.

---

## What's Next

You have built an autoencoder by hand. Now build the family:

- **[Generative → 02 Concepts → VAE](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/generative/02_Concepts.md#vae--encoding-to-a-distribution)** — same encoder/decoder structure plus a KL term that makes the latent space *generative*
- **[Generative → 02 Concepts → GAN](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/generative/02_Concepts.md#gan--two-networks-playing-a-minimax-game)** — replace the reconstruction loss with an adversarial discriminator
- **[Generative → 02 Concepts → Diffusion](https://github.com/sunilmogadati/systems-in-production/blob/main/playbooks/ai/generative/02_Concepts.md#diffusion--iterative-denoising)** — train a U-Net (encoder-decoder with skip connections) to denoise

For full implementations on MNIST in PyTorch:

**[Deep Learning Autoencoders & GANs on Colab](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Deep_Learning_Autoencoders_GANs.ipynb)** — autoencoder, SimpleGAN, DCGAN.